In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datas/btc_4h_features.csv")
df.head()

,timestamp,open,high,low,close,volume,previous_close,tr1,tr2,tr3,true_range,atr_14,future_close,future_return,target,log_return
0,2025-09-18 08:00:00,117086.01,117413.04,117010.59,117063.40,1165.75229,117086.01,402.45,327.03,75.42,402.45,1063.045714,116394.72,-0.005712,0,-0.000193
1,2025-09-18 12:00:00,117063.41,117843.83,116977.59,117583.56,2416.33194,117063.40,866.24,780.43,85.81,866.24,1047.804286,115884.03,-0.014454,0,0.004434
2,2025-09-18 16:00:00,117583.56,117900.00,117196.33,117450.21,1828.62350,117583.56,703.67,316.44,387.23,703.67,1042.707857,115125.12,-0.019796,0,-0.001135
3,2025-09-18 20:00:00,117450.21,117657.17,116612.03,117073.53,1375.21725,117450.21,1045.14,206.96,838.18,1045.14,1033.876429,115632.38,-0.012310,0,-0.003212
4,2025-09-19 00:00:00,117073.53,117459.99,116871.61,116977.58,1326.38784,117073.53,588.38,386.46,201.92,588.38,985.726429,115445.00,-0.013101,0,-0.000820


In [37]:
# ATR percentile (rolling 100 periodi)
df["atr_percentile"] = df["atr_14"].rolling(100).rank(pct=True)

# Volatility ratio
df["vol_ratio"] = df["atr_14"] / df["atr_14"].rolling(50).mean()

# EMA 20
df["ema_20"] = df["close"].ewm(span=20).mean()
df["ema_dist"] = (df["close"] - df["ema_20"]) / df["close"]

df = df.dropna()

df.head()

,timestamp,open,high,low,close,volume,previous_close,tr1,tr2,tr3,true_range,atr_14,future_close,future_return,target,log_return,atr_percentile,vol_ratio,ema_20,ema_dist
99,2025-10-04 20:00:00,121959.25,122410.00,121871.01,122391.00,785.386640,121959.25,538.99,450.75,88.24,538.99,1033.459286,123482.31,0.008917,0,0.003534,0.62,1.087770,120343.673354,0.016728
100,2025-10-05 00:00:00,122390.99,124362.28,122136.00,123776.22,4212.026150,122391.00,2226.28,1971.28,255.00,2226.28,1073.733571,123839.09,0.000508,0,0.011254,0.69,1.131546,120670.595876,0.025091
101,2025-10-05 04:00:00,123776.23,125708.42,123734.00,124713.63,7342.549203,123776.22,1974.42,1932.20,42.22,1974.42,1154.653571,123390.47,-0.010610,0,0.007545,0.92,1.215357,121055.660936,0.029331
102,2025-10-05 08:00:00,124713.63,124838.37,122520.24,123464.64,4455.309640,124713.63,2318.13,124.74,2193.39,2318.13,1268.027143,124215.00,0.006078,0,-0.010065,0.98,1.329203,121285.095164,0.017653
103,2025-10-05 12:00:00,123464.64,123464.65,122675.00,122803.17,2233.090040,123464.64,789.65,0.01,789.64,789.65,1273.435000,125000.00,0.017889,0,-0.005372,0.98,1.329191,121429.678082,0.011184


In [38]:
features = [
    "atr_14",
    "log_return",
    "atr_percentile",
    "vol_ratio",
    "ema_dist"
]

X = df[features]
y = df["target"]

In [39]:
split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [40]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC:", roc_auc_score(y_test, y_proba))

AUC: 0.5873138161273754


In [42]:
df[[
    "atr_percentile",
    "vol_ratio",
    "ema_dist",
    "future_return"
]].corr()

,atr_percentile,vol_ratio,ema_dist,future_return
atr_percentile,1.000000,0.873448,-0.330302,-0.028421
vol_ratio,0.873448,1.000000,-0.356748,0.056871
ema_dist,-0.330302,-0.356748,1.000000,-0.119036
future_return,-0.028421,0.056871,-0.119036,1.000000


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

features = [
    "atr_14",
    "log_return",
    "atr_percentile",
    "vol_ratio",
    "ema_dist"
]

y = df["target"]

for col in features:
    
    X_single = df[[col]]
    
    split_index = int(len(df) * 0.7)
    
    X_train = X_single.iloc[:split_index]
    X_test = X_single.iloc[split_index:]
    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    print(col, "AUC:", round(auc, 4))

atr_14 AUC: 0.6376
log_return AUC: 0.52
atr_percentile AUC: 0.3889
vol_ratio AUC: 0.4094
ema_dist AUC: 0.6281


In [44]:
features = ["atr_14", "ema_dist"]

X = df[features]
y = df["target"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC:", roc_auc_score(y_test, y_proba))

AUC: 0.602336928608115


In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Feature migliori
features = ["atr_14", "ema_dist"]

X = df[features]
y = df["target"]

# Split temporale
split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Random Forest (non serve scaling, ma lo lasciamo coerente)
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:,1]

print("Random Forest AUC:", roc_auc_score(y_test, y_proba))

Random Forest AUC: 0.5666409861325116
